In [ ]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="""
        LangGraph is a framework for building stateful applications
        with large language models. It allows developers to represent
        application workflows as graphs containing nodes and edges.
        """,
        metadata={"topic": "LangGraph"}
    ),

    Document(
        page_content="""
        LangGraph StateGraph allows developers to define a shared state
        that is passed between different nodes in a graph. Each node can
        read and update the state.
        """,
        metadata={"topic": "StateGraph"}
    ),

    Document(
        page_content="""
        LangGraph supports persistence. Checkpointers can save graph
        state during execution, allowing applications to resume from
        previous states and maintain conversation memory.
        """,
        metadata={"topic": "Persistence"}
    ),

    Document(
        page_content="""
        LangGraph supports human-in-the-loop workflows. A graph can
        pause execution and request human approval before continuing.
        """,
        metadata={"topic": "Human-in-the-loop"}
    ),

    Document(
        page_content="""
        LangGraph can be used to implement agentic RAG systems.
        A graph can contain retrieval nodes, LLM nodes, tools and
        conditional routing logic.
        """,
        metadata={"topic": "RAG"}
    )
]

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOllama(model="llama3.2", temperature=0)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

hyde_prompt = ChatPromptTemplate.from_template("""
You are helping a RAG system retrieve relevant documents.

Given the user's question, generate a hypothetical document
that would contain the information needed to answer the question.

The document does not need to be factually correct.
Its purpose is to contain terminology and concepts that are
semantically similar to the documents that may exist in the
knowledge base.

Do not mention that this is a hypothetical document.

User question:
{question}

Generate only the hypothetical document.
""")

hyde_chain = hyde_prompt | llm

In [ ]:
question = """
How can an application remember previous conversations
using LangGraph?
"""

response = hyde_chain.invoke({
    "question": question
})

print(response.content)